# Evidence Retrieval Walkthrough

Thin notebook over the `evidence_retrieval` package. For metrics, ablations, and the
full case study, see the root **README** and `python scripts/run_retrieval_eval.py`.

**Not a fact-checker.** ISOT labels are source buckets; nearest neighbors ≠ verification.

## Build or load an index

In [ ]:
from pathlib import Path
from evidence_retrieval.index import IndexConfig, PassageIndex

INDEX_DIR = Path("data/retrieval_index/default")

if (INDEX_DIR / "chunks.parquet").exists():
    index = PassageIndex.load(INDEX_DIR)
    print(f"Loaded index: {len(index.chunks):,} passages")
else:
    config = IndexConfig(n_articles=2000, chunk_words=120, fields=("body",))
    index = PassageIndex.build(data_dir="data", config=config, show_progress=True)
    index.save(INDEX_DIR)
    print(f"Built + saved index → {INDEX_DIR}")

## Query by claim / article text

In [ ]:
query = "Federal Reserve interest rate decision and markets"
hits = index.query_df(query, top_k=5, method="hybrid")
display(hits[["rank", "score", "label_name", "title", "passage"]])

## Optional: classify-then-retrieve vs retrieve-first

Uses a TF-IDF classifier trained off the indexed sample, then contrasts predicted
source-bucket vs neighborhood vote among retrieved passages. Still not a fact-check.

In [ ]:
from evidence_retrieval.workflows import compare_workflows

comparison = compare_workflows(index, data_dir="data", n_demo=6)
display(comparison)

## Reproduce the README metrics table

```bash
python scripts/run_retrieval_eval.py
```